# TRELLIS.2 — Visible Drive Batch Image → 3D

Put images here:
**`My Drive/Shared/Trellis/Input image`**

3D models are saved here:
**`My Drive/Shared/Trellis/Output`**

Then choose a GPU runtime and click **Runtime → Run all**.

This version deliberately shows what is happening: installer stages, Hugging Face authentication, every TRELLIS/DINO/RMBG model selected for download, file/byte download progress, model loading, internal TRELLIS inference stages, texture baking, GLB export, and per-image timing.


## 1 — Mount Drive + get TRELLIS helpers


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import getpass, os, shutil, subprocess, time

drive.mount('/content/drive', force_remount=False)
ROOT = Path('/content/drive/MyDrive/Shared/Trellis')
INPUT_DIR = ROOT / 'Input image'
OUTPUT_DIR = ROOT / 'Output'
DONE_DIR = ROOT / 'Done'
FAILED_DIR = ROOT / 'Failed'
CACHE_DIR = ROOT / 'cache'
for folder in (INPUT_DIR, OUTPUT_DIR, DONE_DIR, FAILED_DIR, CACHE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print('📥 Input :', INPUT_DIR, flush=True)
print('📤 Output:', OUTPUT_DIR, flush=True)

REPO = Path('/content/My-works')
if REPO.exists():
    shutil.rmtree(REPO)
print('\n[SETUP] Cloning latest My-works helpers...', flush=True)
subprocess.run(['git','clone','--depth','1','https://github.com/Logan17de/My-works.git',str(REPO)], check=True)
TOOLS_3D = REPO / 'ai-3d-animation-engines' / '3d-engine'
os.environ['ENGINE_CACHE_ROOT'] = str(CACHE_DIR)
print('[SETUP] ✅ Helpers ready:', TOOLS_3D, flush=True)


## 2 — Install TRELLIS.2

The installer prints each dependency/build stage live below.


In [ ]:
print('=' * 78, flush=True)
print('TRELLIS.2 INSTALLATION — LIVE OUTPUT', flush=True)
print('=' * 78, flush=True)
started = time.time()
subprocess.run(['bash', str(TOOLS_3D / 'install_3d.sh')], check=True)
print(f'\n[INSTALL] ✅ Finished in {(time.time()-started)/60:.1f} min', flush=True)


## 3 — Hugging Face auth + visible model downloads

This cell prints the exact repositories/files TRELLIS needs and shows Hugging Face/Xet download progress. If a file is already cached it will finish immediately.


In [ ]:
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Hugging Face READ token: ').strip()
if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN is required.')

env = os.environ.copy()
env['HF_TOKEN'] = HF_TOKEN
env['HF_HOME'] = '/content/huggingface'
env['HF_XET_HIGH_PERFORMANCE'] = '1'
env['PYTHONUNBUFFERED'] = '1'
env.pop('HF_HUB_DISABLE_PROGRESS_BARS', None)

TRELLIS_PYTHON = '/opt/conda/envs/trellis2/bin/python'
print('=' * 78, flush=True)
print('MODEL DOWNLOADS — LIVE OUTPUT', flush=True)
print('Expected model groups:', flush=True)
print('  1. microsoft/TRELLIS.2-4B', flush=True)
print('  2. microsoft/TRELLIS-image-large', flush=True)
print('  3. facebook/dinov3-vitl16-pretrain-lvd1689m', flush=True)
print('  4. briaai/RMBG-2.0', flush=True)
print('=' * 78, flush=True)
started = time.time()
subprocess.run([TRELLIS_PYTHON, str(TOOLS_3D / 'prepare_hf_models.py')], cwd='/content/TRELLIS.2', env=env, check=True)
print(f'\n[MODELS] ✅ All required files ready in {(time.time()-started)/60:.1f} min', flush=True)


## 4 — Process every image until the input folder is empty

The TRELLIS model loads once. For each image you will see background removal, DINOv3 encoding, sparse structure generation, shape generation/refinement, texture generation, mesh/PBR decode, remesh/UV/texture bake, and GLB export.


In [ ]:
print('=' * 78, flush=True)
print('TRELLIS BATCH GENERATION — LIVE OUTPUT', flush=True)
print('Input :', INPUT_DIR, flush=True)
print('Output:', OUTPUT_DIR, flush=True)
print('=' * 78, flush=True)
subprocess.run([TRELLIS_PYTHON, str(TOOLS_3D / 'batch_drive_trellis2.py')], cwd='/content/TRELLIS.2', env=env, check=True)
